# Día 4 — Exploración de arquitecturas RX

Comparación **CNN propia** vs **Transfer Learning** (ResNet50, EfficientNetB0, DenseNet121).

**Decisión del proyecto:** Transfer Learning con **ResNet50** (ver `docs/ml/arquitectura-rx.md`).

Este notebook no entrena el modelo final; prepara el Día 5.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from models.architectures import (
    LABELS,
    build_baseline_cnn,
    build_transfer_classifier,
    compare_architectures,
)

FEATURES_DIR = ROOT.parent / "data" / "processed" / "features" / "v1"
TRAIN_DIR = FEATURES_DIR / "train"
print("Features:", FEATURES_DIR)
print("Existe:", FEATURES_DIR.exists())

In [ ]:
def sample_paths(n_per_label: int = 2) -> list[Path]:
    paths = []
    for label in LABELS:
        folder = TRAIN_DIR / label
        if not folder.is_dir():
            continue
        for p in sorted(folder.glob("*.jpg"))[:n_per_label]:
            paths.append(p)
    return paths

samples = sample_paths()
if not samples:
    raise FileNotFoundError(
        "No hay imágenes en features/v1/train. Ejecuta preprocess_images (Día 3)."
    )

fig, axes = plt.subplots(1, len(samples), figsize=(3 * len(samples), 3))
if len(samples) == 1:
    axes = [axes]
for ax, path in zip(axes, samples):
    ax.imshow(Image.open(path))
    ax.set_title(path.parent.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame(compare_architectures())
comparison

In [ ]:
baseline = build_baseline_cnn()
resnet = build_transfer_classifier("resnet50", trainable_backbone=False)

print("=== Baseline CNN ===")
baseline.summary()
print("\n=== ResNet50 (modelo elegido) ===")
resnet.summary()

In [ ]:
import numpy as np

batch = np.stack(
    [np.array(Image.open(p).resize((224, 224))) for p in samples[:3]],
    axis=0,
)
probs = resnet.predict(batch, verbose=0)
for path, prob in zip(samples[:3], probs):
    pred = LABELS[int(np.argmax(prob))]
    print(f"{path.name} ({path.parent.name}) -> {pred} (pesos aleatorios, sin entrenar)")

## Conclusión

- **Modelo final:** `build_transfer_classifier(backbone="resnet50")`.
- **Día 5:** entrenar con `image_dataset_from_directory`, checkpoints, métricas y matriz de confusión.
- Documento completo: `docs/ml/arquitectura-rx.md`.